In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("PostgresConnector") \
    .config("spark.jars","/usr/lib/spark/jars/postgresql-42.7.4.jar")\
    .getOrCreate()


In [ ]:
# Question 53: Create a unified list of all current Employees and all current Managers.
# Concept: UNION / Union All
employee_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.employee",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

department_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

department_employee_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department_employee",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})
department_manager_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department_manager",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})
# Solution:
emp = employee_df.join(department_employee_df, employee_df.id == department_employee_df.employee_id) \
                 .filter(col("to_date") == '9999-01-01') \
                 .select(employee_df.id, employee_df.first_name, lit("Employee").alias("type"))

mgr = employee_df.join(department_manager_df, employee_df.id == department_manager_df.employee_id) \
                 .filter(col("to_date") == '9999-01-01') \
                 .select(employee_df.id, employee_df.first_name, lit("Manager").alias("type"))

emp.union(mgr).show()
